# 30 — Umpire tiers

Five performance tiers for every season, 2021–2026. All the logic lives in `src/umpire_tiers.py`; this notebook runs it and shows the results.

**Ground rule.** Each season is graded against its own strike zone (old zone 2021–2025, new ABS zone 2026). Nothing is standardised across seasons, so an umpire is only compared with umpires who called the same zone in the same year.

**How a tier is assigned**

1. **Accuracy above league (AAL).** The umpire's share of correct calls minus the share for all qualifying umpires that season, in percentage points (pp).
2. **Shrinkage.** A number built on 11 games is noisier than one built on 31. Each AAL is pulled toward zero by its reliability: `shrunken AAL = AAL × skill² / (skill² + se²)`. `se` is the umpire's own noise, measured from his game-to-game variation; `skill` is how much umpires truly differ that season.
3. **Tiers.** Umpires are ranked on the shrunken AAL within the season. Top 10% = Tier 1 (Elite), next 15% = Tier 2 (High), middle 50% = Tier 3 (Average), next 15% = Tier 4 (Below average), bottom 10% = Tier 5 (Lowest). Tiers 1 + 2 are the top quartile; Tiers 4 + 5 are the bottom quartile.

Umpires with fewer than 10 plate games stay in the table, flagged `qualifies = False`, without a tier.

In [ ]:
import sys

import pandas as pd
from IPython.display import display

sys.path.insert(0, "../src")
import umpire_tiers

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 40)

tables = umpire_tiers.build()
umpire_tiers.save(tables)

tiers = tables["umpire_tiers"]
summary = tables["tier_summary"]
career = tables["umpire_career_tiers"]
stability = tables["tier_stability_summary"]
pool = tiers[tiers["qualifies"]]
print(len(tiers), "umpire-seasons |", len(pool), "qualify (10+ plate games)")

## 1. The league each season

`skill_sd_pp` is how much umpires truly differ once luck is removed. `typical_se_pp` is the noise in one umpire's season. `typical_shrink` is the share of his gap to the league that a typical umpire keeps.

In [ ]:
overview = pool.groupby("season").agg(
    zone=("zone", "first"),
    umpires=("official_id", "count"),
    league_accuracy=("league_accuracy", "first"),
    skill_sd_pp=("skill_sd_pp", "first"),
    typical_se_pp=("se_game_pp", "median"),
    typical_shrink=("shrink_factor", "median"),
)
overview.round(4)

## 2. What each tier looks like

`lowest_aal_shrunk_pp` is the cut-off to get into the tier. `misses_per_game` is the fan number: wrong ball/strike calls per game behind the plate.

In [ ]:
SEASON = 2026
columns = [
    "tier",
    "tier_name",
    "umpires",
    "lowest_aal_shrunk_pp",
    "highest_aal_shrunk_pp",
    "accuracy",
    "misses_per_1000",
    "misses_per_game",
]
summary.loc[summary["season"] == SEASON, columns].round(3)

In [ ]:
print("Accuracy by tier, every season")
display(summary.pivot(index="season", columns="tier", values="accuracy").round(4))
print("Misses per game by tier, every season")
display(
    summary.pivot(index="season", columns="tier", values="misses_per_game").round(2)
)

## 3. Tier 1 and Tier 5 in 2026

`secure` = the umpire's interval does not include the league average, so his place above or below the league is not luck.

In [ ]:
show = [
    "rank_in_season",
    "official_name",
    "plate_games",
    "called_pitches",
    "missed",
    "accuracy",
    "aal_pp",
    "shrink_factor",
    "aal_shrunk_pp",
    "shrunk_low_pp",
    "shrunk_high_pp",
    "secure",
    "misses_per_game",
]
season = pool[pool["season"] == SEASON]
print("Tier 1 — Elite")
display(season.loc[season["tier"] == 1, show].round(3))
print("Tier 5 — Lowest")
display(season.loc[season["tier"] == 5, show].round(3))

## 4. The full 2026 ranking

In [ ]:
season[
    [
        "rank_in_season",
        "official_name",
        "tier",
        "tier_name",
        "plate_games",
        "accuracy",
        "aal_shrunk_pp",
        "percentile",
        "secure",
    ]
].round(3)

## 5. Career tiers: three or more qualifying seasons

`career_score_pp` is the average of an umpire's shrunken AAL over his qualifying seasons. The old-zone seasons (2021–2025) and the new-zone season (2026) are shown in separate columns so the two zones are never blended out of sight.

In [ ]:
career_columns = [
    "career_rank",
    "official_name",
    "seasons",
    "career_score_pp",
    "career_se_pp",
    "old_zone_score_pp",
    "new_zone_score_pp",
    "tiers_by_season",
    "career_tier_name",
]
eligible = career[career["eligible"]]
print(len(eligible), "umpires with 3+ qualifying seasons")
print("Top 10")
display(eligible[career_columns].head(10).round(3))
print("Bottom 10")
display(eligible[career_columns].tail(10).round(3))

## 6. Do tiers hold from one season to the next?

If tiers measured luck, an umpire's tier this year would say nothing about next year. The `by_chance` columns show what luck alone would give.

In [ ]:
stability.round(3)

## Takeaways

1. **How a tier is made.** Accuracy above the league that season, pulled toward the league when the sample is small, then cut at the 10th / 25th / 75th / 90th percentiles. 524 umpire-seasons qualify; 38 with under 10 plate games stay in the table, flagged.
2. **The 2026 tiers in fan numbers.** Tier 1 (9 umpires): 94.6% correct, 8.2 missed calls a game. Tier 3: 93.5%, 10.0. Tier 5 (9 umpires): 92.4%, 11.8. Tier 1 against Tier 5 is 2.2 pp: 22 fewer misses per 1,000 calls, about 3.5 a game, about 100 a season.
3. **2026 cut-offs** (shrunken AAL): Tier 1 from +0.63 pp, Tier 2 from +0.44, Tier 3 from −0.36 to +0.41, Tier 4 from −0.64, Tier 5 below that.
4. **Tier 1, 2026:** Visconti, Ripperger, Conroy, D. Wilson, Morales, Krupa, Moscoso, T. Jones, Valentine (8 of 9 secure). **Tier 5:** M. Moore, Kulpa, A. Johnson, Estabrook, Barber, Pawol, Fletcher, Bacon, Dreckman (5 of 9 secure).
5. **The gap between umpires is closing.** True-skill spread fell from 0.78 pp (2021) to 0.58 pp (2026). Tier 5 went from 90.5% to 92.4% while Tier 1 went from 93.6% to 94.6%: the bottom improved about twice as much as the top. (2026 is the new zone; each season is on its own zone.)
6. **Career, 89 umpires with 3+ seasons.** Top: Pat Hoberg (Tier 1 in all three of his seasons), John Libka, Alex Tosi, Derek Thomas, Jeremie Rehak, Alan Porter, Quinn Wolcott, Jansen Visconti, Mark Ripperger. Bottom: CB Bucknor, Rob Drake, Hunter Wendelstedt, Laz Diaz, Ron Kulpa, Larry Vanover, Brian O'Nora, Andy Fletcher, Bruce Dreckman.
7. **Tiers are skill, not luck.** 49% of umpires are in the same tier the next season (32% by chance) and 88% are within one tier (68% by chance). Year-to-year correlation of AAL is 0.62, and 0.65 across the change of zone from 2025 to 2026: the ranking survived the new zone.
8. **Pitches in a game are close to independent.** The game-based noise is only 1–10% larger than the simple binomial noise.

Outputs in `reports/tables`: `umpire_game_accuracy.csv`, `umpire_tiers.csv`, `tier_summary.csv`, `umpire_career_tiers.csv`, `tier_stability.csv`, `tier_stability_summary.csv`.